# Fase 0: Introduzione


In questo Jupyter Notebook sarà possibile consultare la realizzazione del primo obiettivo del Project Work "Triage automatico dei ticket con Machine Learning". Gli obiettivi raggiunti sono:
- *Creazione di un piccolo dataset sintetico (200–500 ticket testuali) con etichetta di categoria e priorità*;
    - Sintesi di ticket brevi (titolo + descrizione) con lessico tipico per Amministrazione, Tecnico, Commerciale;
    - Etichette di priorità semplici basate su parole chiave (es. 'errore', 'fattura', 'bloccante');
    - Esportazione CSV con colonne: id, title, body, category, priority.


Tengo a precisare che il contenuto di questo Notebook, nonostante non molto innovativo e molto meno moderno rispetto alla generazione sintetica dei dati trattati all'interno del secondo Jupyter Notebook, contiene comunque elementi che si sono rivelati essenziali per la selezione delle migliori scelte di sviluppo da utilizzare per il progetto. Per questo motivo ho deciso di lasciarlo comunque parte dell'elaborato finale.

Nello specifico, viene presentata un'analisi dei processi che hanno portato alla creazione di un dataset pseudorandomico che, come verrà analizzato nel notebook, è sì un ottimo punto di partenza, però sarà necessario sviluppare qualcosa di più avanzato. Il percorso è composto da 6 fasi:
- Fase 1: Caricamento e pulizia dei dati
    - Per la generazione del dataset tramite codice è fondamentale l'utilizzo della pseudo-casualità. Per comprendere meglio questa tecnica, ho deciso di iniziare dalla creazione di finti utenti realistici. Partendo da liste di nomi e cognomi reali, lo script genera combinazioni creando falsi utenti e relative email. Ho deciso di mantenere questo output nonostante non fosse richiesto dalla consegna per rendere il dataset realizzato più realistico. In questa fase, i dati in input vengono estratti e riordinati dai file "nomi_f.csv", "nomi_m.csv" e "cognomi.csv".
- Fase 2: Generazione pool di utenti (nome ed email)
    - In questa seconda fase viene illustrato il processo di generazione degli utenti come descritto nella descrizione della Fase 1.
- Fase 3: Generazione ticket brevi con lessico tipico casuale
    - Qui si entra nel core del progetto: ho sfruttato le meccaniche sviluppate nella fase 2 per creare un dataset elementare con lessico tipico casuale
- Fase 4: Esportazione e analisi del file dei ticket
    - In questa sezione è possibile vedere ed analizzare il risultato della fase 3. Come verrà descritto, il dataset elementare creato nella fase precedente è valido, ma mostra ampi margini di miglioramento
- Fase 5: Ottimizzazione del dataset
    - Vengono implementate ed esaminate le ottimizzazioni e i correttivi discussi nella fase precedente.
- Fase 6: Generazione dataset avanzato tramite LLM
    - Qui ho sfruttato le potenzialità di Gemma4 (un modello di LLM utilizzabile in locale senza limiti di utilizzi) per creare un ulteriore dataset ancora più verosimile. L'intero codice è consultabile presso il Notebook 2: Generazione Dataset Avanzato con Gemma
- Fase 7: Analisi finale dei dataset elementari
    - Un confronto diretto tra i diversi dataset elementari generati lnel Notebook
 

_Installazione di librerie utilizzate nel progetto:_ 

In [13]:
%pip install pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


# Fase 1: Caricamento e pulizia dei dati

Come prima cosa, il codice legge in input i file contenenti i nomi e i cognomi, con lo scopo di creare degli utenti verosimili. I tre file utilizzati sono stati scaricati sotto forma di txt al seguente link: https://github.com/Max1234-Ita/Liste. Sono stati convertiti in formato csv per dare una struttura più ordinata e leggibile.

In [14]:
import pandas as pd
df_nomi_f = pd.read_csv(
    "input/nomi_f.csv", comment="#", header=None, names=["Nome", "Rarità"]
)
df_nomi_m = pd.read_csv(
    "input/nomi_m.csv", comment="#", header=None, names=["Nome", "Rarità"]
)
df_cognomi = pd.read_csv("input/cognomi.csv", comment="#", header=None, names=["Cognome"])
df_nomi = pd.concat([df_nomi_f, df_nomi_m], ignore_index=True)
df_nomi.sample(10)

,Nome,Rarità
2587,MARLENA,2
8348,ROSAURO,4
3693,SANTA,3
6666,ITO,4
200,AMABILIA,4
8315,ROMERO,4
4333,ALBERICO,2
2060,HANANIA,3
7818,PERFETTO,4
3087,ONESINA,4


A questo punto, ho a disposizione due diversi DataFrame:
- df_nomi: contiene una lista di nomi italiani e la loro rarità
- df_cognomi: contiene una lista con cognomi italiani

# Fase 2: Generazione pool di utenti (nome ed email)

Partendo dai due DataFrame, possiamo adesso creare una lista di utenti aventi nomi ed email fittizie.
Il primo problema da affrontare è trovare un meccanismo per rendere i nomi più rari meno presenti all'interno della pool. Per fare questo, ho assegnato un peso probabilistico alle varie categorie. Stessa sorte va anche ad una pool di domini email.

Questa cosa concede un maggiore livello di pseudorandomicità, facendo sì che ad esempio le email *gmail.com siano più frequenti rispetto alle email *yahoo.com.

In [15]:
categorie_nomi = [1, 2, 3, 4]
pesi_categorie_nomi = [0.85, 0.11, 0.03, 0.01]
domini = ["gmail.com", "outlook.it", "hotmail.com", "libero.it", "yahoo.com"]
pesi_domini = [0.55, 0.15, 0.13, 0.13, 0.04]

Altro elemento frequente nelle email è la data di nascita dell'utente. La funzione *genera_anno*, usando *random.randint()*, sorteggia un anno casuale. Questa verrà usata durante la creazione dei vari utenti per decidere il loro anno di nascita.

In [16]:
import random
def genera_anno():
    anno = random.randint(1970, 2007)
    return str(anno)[2:]

In questo ciclo, vengono estratti casualmente nomi, cognomi, domini per essere combinati dinamicamente seguendo diversi pattern strutturali; questo piccolo dettaglio rende le email meno statiche e più realistiche.
Da notare inoltre che, nell'estrazione del nome, non vi è una totale casualità grazie al controllo dei pesi (*weights*).

A valle, troviamo la popolazione di una lista *pool_utenti* che conterrà *numero_utenti_unici* nomi inventati e *numero_utenti_unici* email corrispettive.

In [17]:
numero_utenti_unici = 125
pool_utenti = []
for _ in range(numero_utenti_unici):
    cat_nome = random.choices(categorie_nomi, weights=pesi_categorie_nomi)[0]
    df_nomi_filtrato = df_nomi[df_nomi["Rarità"] == cat_nome]
    if df_nomi_filtrato.empty:
        df_nomi_filtrato = df_nomi

    # Conserviamo una versione "pulita" con le iniziali maiuscole
    nome_reale = df_nomi_filtrato["Nome"].sample(n=1).values[0].title().strip()
    cognome_reale = df_cognomi["Cognome"].sample(n=1).values[0].title().strip()
    nome_completo = f"{nome_reale} {cognome_reale}"

    # Versione minuscola e senza spazi per l'indirizzo email
    nome_email = nome_reale.lower().replace(" ", "").replace("'", "")
    cognome_email = cognome_reale.lower().replace(" ", "").replace("'", "")

    dominio = random.choices(domini, weights=pesi_domini)[0]
    anno_nascita = genera_anno()

    formati = [
        f"{nome_email}.{cognome_email}",
        f"{nome_email}.{cognome_email}{anno_nascita}",
        f"{nome_email[0]}.{cognome_email}",
        f"{nome_email[0]}.{cognome_email}{anno_nascita}",
        f"{cognome_email}.{nome_email}",
        f"{nome_email}_{cognome_email}",
        f"{nome_email}{cognome_email}{anno_nascita}",
    ]
    pesi_formati = [0.20, 0.40, 0.10, 0.10, 0.10, 0.05, 0.05]
    struttura_email = random.choices(formati, weights=pesi_formati)[0]
    email_finale = f"{struttura_email}@{dominio}"

    pool_utenti.append({"user_name": nome_completo, "user_email": email_finale})

# Fase 3: Generazione ticket brevi con lessico tipico casuale

In questa fase avviene la generazione sintetica di dataset brevi ed elementari.
Per farlo, ho creato un file JSON contenente un dizionario lessicale (che può facilmente essere esteso o adattato a piacimento). Il file in questione si trova all'interno della cartella *input => lessico.json*.

Il JSON contiene una divisione per 3 tipologie di ticket, come da consegna:
- Amministrazione
- Tecnico
- Commerciale

Ogni divisione contiene a sua volte 3 liste, ovvero:
- Verbi
- Sostantivi
- Dettagli del ticket (frasi o contesti specifici)

La funzione *genera_ticket_base* leggerà il JSON e sarà in grado di creare *num_ticket* tickets, estraendo un utente casuale di volta in volta.

In maniera del tutto casuale, lo script seleziona poi una categoria ed estrae un verbo, un sostantivo e dei dettagli. Questi verranno combinati in un body, estratto anch'esso casualmente da una lista contenenti diversi template, con lo scopo di creare dei ticket sempre diversi.

Il livello della priorità viene deciso di conseguenza in base al numero di parole in base all'urgenza che queste esprimono, anticipando un po' il processo inverso che verrà poi trattato dagli algoritmi di Machine Learning nella pipeline.

Infine, la funzione organizza questo insieme di dati estratti casualmente in un unico record dove troviamo:
- id: valore incrementale che funge da chiave primaria univoca
- user_name: nome e cognome di un utente estratto casualmente dalla pool creata nella fase 2
- user_email: indirizzo email associato al medesimo utente
- title: oggetto del ticket
- body: il corpo effettivo del ticket
- category: categoria del ticket (Amministrazione, Tecnico o Commerciale)
- priority: il livello di urgenza

In [18]:
import json
import random
import pandas as pd


def genera_ticket_base(nome_file_input, pool_utenti, num_ticket=200,nome_file_output="dataset/ticket_utenti_simulati.csv"):
    with open(nome_file_input, "r", encoding="utf-8") as f:
        lessico = json.load(f)

    categorie = list(lessico.keys())
    # Keywords più importanti
    keywords_alta = {
        "bloccante",
        "offline",
        "crash",
        "urgente",
        "aiuto",
        "leak",
        "picco di traffico",
        "scaduto",
        "sospendere",
        "penale",
    }
    keywords_media = {
        "errore",
        "errato",
        "sollecitare",
        "sollecito",
        "lento",
        "bug",
        "mancante",
        "disdire",
        "rettificare",
        "stornare",
        "problema",
    }

    pesi_oggetti = [0.20,0.25,0.10,0.15,0.05,0.10,0.10,0.05]
    pesi_formati = [0.15,0.20,0.15,0.10,0.05,0.10,0.10,0.11,0.01,0.01,0.01,0.01,0.01]

    utenti_estratti = random.choices(pool_utenti, k=num_ticket)

    dataset = []

    for i in range(num_ticket):
        utente = utenti_estratti[i]
        cat = random.choice(categorie)

        # Estrazione casuale delle componenti testuali
        v = random.choice(lessico[cat]["verbi"])
        s = random.choice(lessico[cat]["sostantivi"])
        d = random.choice(lessico[cat]["dettagli"])

        # Template per gli oggetti
        oggetti = [
            f"{v.capitalize()} {s}",
            f"URGENTE: {v.capitalize()} {s}",
            f"Richiesta supporto - categoria: {cat}",
            f"Segnalazione {s}",
            f"AIUTO - problema con {s}",
            f"Ticket {v.capitalize()} {s}",
            f"Richiesta info {s} {d}",
            f"{s} - {d}",
        ]
        title = random.choices(oggetti, weights=pesi_oggetti)[0]

        # Template per i corpi del testo
        formati = [
            f"Buongiorno,con la presente si richiede cortesemente di {v} {s} {d}.Restiamo in attesa di un vostro riscontro urgente.Cordiali saluti.",
            f"Ciao, per favore dovremmo {v} {s} {d} il prima possibile.Fammi sapere si ci sono problemi. Grazie!",
            f"Salve, si richiede di {v} {s} {d}. Restiamo in attesa di un riscontro urgente.",
            f"Potete {v} {s} {d}?Grazie, saluti.",
            f"[NOTIFICA AUTOMATICA]: Rilevata necessità di {v} {s} {d}. Si prega di non rispondere a questa email.",
            f"Gentili colleghi,vi ricontatto per sapere se ci sono novità in merito alla richiesta di {v} {s} {d}.Rimango a disposizione.Buon lavoro.",
            f"Purtroppo si è reso necessario {v} {s} {d}.Procedere appena possibile, blocca il lavoro del team.Grazie.",
            f"ciao, bisogna {v} {s} {d} entro oggi se riusciamo... fatemi sapere, grazie",
            f"Per curiosità stavo spulciando {s}, ho scoperto che {v} {s} {d}, e questo è un problema!",
            "ho un problema, quando posso chiamarvi?",
            f"Salve, sono {utente['user_name']}. Ho scoperto che {v} {s}:{d}",
            f"{v} {s} {d}.",
            f"sono {utente['user_name']}, ho un problema critico, ne possiamo parlare il prima possibile?",
        ]
        body = random.choices(formati, weights=pesi_formati)[0]

        testo_completo = f"{title} {body}".lower()

        # Controlla se la keyword è presente come sottostringa nel testo
        conteggio_alta = sum(1 for kw in keywords_alta if kw in testo_completo)
        conteggio_media = sum(1 for kw in keywords_media if kw in testo_completo)

        if conteggio_alta >= 2 and conteggio_media <= 1:
            priority = "Alta"
        elif conteggio_media >= 2 and conteggio_alta <= 1:
            priority = "Media"
        elif conteggio_alta > conteggio_media:
            priority = "Alta"
        elif conteggio_media > 0:
            priority = "Media"
        else:
            priority = "Bassa"

        dataset.append(
            {
                "id": f"TKT-{i+1:04d}",
                "user_name": utente["user_name"],
                "user_email": utente["user_email"],
                "title": title,
                "body": body,
                "category": cat,
                "priority": priority,
            }
        )

    # Scrittura finale ottimizzata
    df = pd.DataFrame(dataset)
    df.to_csv(nome_file_output, index=False, encoding="utf-8-sig")

    print(
        f"Successo! Creato il file '{nome_file_output}' con {num_ticket} righe totali."
    )

# Fase 4: Esportazione e analisi del file dei ticket

Mandiamo in esecuzione lo script descritto nella fase 3. L'output sarà la creazione di un file contenente utenti e ticket generati in maniera pseudocasuale dentro la cartella *dataset => ticket_pseudorandom.csv*.

Se lo si desidera, è possibile mandare in esecuzione questa cella per la generazione di un nuovo dataset e per vedere a video il risultato.

In [19]:
genera_ticket_base("input/lessico.json", pool_utenti, 500,"dataset/ticket_pseudorandom.csv")
df_anteprima = pd.read_csv("dataset/ticket_pseudorandom.csv")
print("*"*10)
df_anteprima.head(5)["body"]
print("*"*10)
print(df_anteprima["priority"].value_counts())

Successo! Creato il file 'dataset/ticket_pseudorandom.csv' con 500 righe totali.
**********
**********
priority
Alta     223
Media    140
Bassa    137
Name: count, dtype: int64


Guardando i risultati è evidente che la generazione pseudocasuale dia un risultato discreto, ma si può fare di meglio. Il rischio attuale è che utilizzare un dataset con una struttura così rigida e con poche varianti porti un qualsiasi algoritmo di Machine Learning a memorizzare passivamente combinazioni fisse di parole (overfitting).

*Il successivo sviluppo della Pipeline ha confermato questo dubbio. La precisione risultava altissima, ma il modello entrava in difficolta quando venivano utilizzate parole proveniente da un vocabolario diverso o strutture lessicali diverse. Ho inizialmente provato a mettere una pezza introducendo ambiguità nel testo, creando delle frasi che introducano una percentuale di ticket con un vocabolario misto o simulando l'errore umano inserendo rumore (typo, refusi nel testo e diverso registro linguistico).*

# Fase 5: Ottimizzazione del dataset

Il primo problema affrontato in questa fase è stato quello della generazione del rumore (*Noise Injection*). Ad occuparsene c'è la funzione 'introduci_typo', che prende in input un testo e sceglie casualmente un typo da inserire (come una ripetizione di lettere o un'omissione).

Il secondo problema affrontato è stato l'aggiunta di ambiguità semantica. Adesso c'è una probabilità del 15% che il ticket usi parole estratte da categorie diverse rispetto a quella di appartenenza. Questo significa che il ticket di una categoria (ad esempio Tecnico) ha una piccola probabilità di contenere anche parole appartenenti ad altre categorie.

Infine, ho arricchito i vettori 'oggetti' e 'formati' inserendo espressioni con nuovi registri linguistici per creare maggiore varietà.

In [20]:
import json
import random
import pandas as pd


def introduci_typo(testo, probabilita=0.01):
    if random.random() > probabilita:
        return testo

    parole = testo.split()
    if not parole:
        return testo

    # Sceglie una parola a caso su cui applicare il refuso
    idx_parola = random.randint(0, len(parole) - 1)
    parola = parole[idx_parola]

    if len(parola) > 3:
        modalita = random.choice(
            ["inversione", "omissione", "ripetizione", "case"]
        )

        if modalita == "inversione":
            # (es. "errore" -> "erroer")
            idx = random.randint(1, len(parola) - 2)
            parola = (
                parola[:idx] + parola[idx + 1] + parola[idx] + parola[idx + 2 :]
            )
        elif modalita == "omissione":
            # (es. "backup" -> "bakup")
            idx = random.randint(1, len(parola) - 1)
            parola = parola[:idx] + parola[idx + 1 :]
        elif modalita == "ripetizione":
            # (es. "server" -> "servver")
            idx = random.randint(0, len(parola) - 1)
            parola = parola[:idx] + parola[idx] + parola[idx:]
        elif modalita == "case":
            # (es. "SSL" -> "Ssl" o "errore" -> "eRRORE")
            parola = parola.swapcase()

    parole[idx_parola] = parola
    return " ".join(parole)


def genera_ticket_migliorato(nome_file_input, pool_utenti, num_ticket=200,nome_file_output="dataset/ticket_utenti_simulati_pro.csv"):
    with open(nome_file_input, "r", encoding="utf-8") as f:
        lessico = json.load(f)

    categorie = list(lessico.keys())

    keywords_alta = {
        "bloccante",
        "offline",
        "crash",
        "urgente",
        "aiuto",
        "leak",
        "picco di traffico",
        "scaduto",
        "sospendere",
        "penale",
    }
    keywords_media = {
        "errore",
        "errato",
        "sollecitare",
        "sollecito",
        "lento",
        "bug",
        "mancante",
        "disdire",
        "rettificare",
        "stornare",
        "problema",
    }

    # Distribuzione dei pesi dei template (leggermente aggiornati per i nuovi formati)
    pesi_oggetti = [0.18,0.22,0.10,0.12,0.05,0.10,0.10,0.05,0.08]
    pesi_formati = [0.12,0.15,0.12,0.08,0.05,0.08,0.08,0.08,0.02,0.02,0.02,0.02,0.02,0.07,0.07]

    utenti_estratti = random.choices(pool_utenti, k=num_ticket)
    dataset = []

    for i in range(num_ticket):
        utente = utenti_estratti[i]
        cat_reale = random.choice(categorie)

        # C'è una probabilità del 15% che il ticket usi parole incrociate da altre categorie
        cat_verbo = (
            cat_reale if random.random() > 0.15 else random.choice(categorie)
        )
        cat_sost = (
            cat_reale if random.random() > 0.15 else random.choice(categorie)
        )
        cat_dett = (
            cat_reale if random.random() > 0.15 else random.choice(categorie)
        )

        v = random.choice(lessico[cat_verbo]["verbi"])
        s = random.choice(lessico[cat_sost]["sostantivi"])
        d = random.choice(lessico[cat_dett]["dettagli"])

        oggetti = [
            f"{v.capitalize()} {s}",
            f"URGENTE: {v.capitalize()} {s}",
            f"Richiesta supporto - categoria: {cat_reale}",
            f"Segnalazione {s}",
            f"AIUTO - problema con {s}",
            f"Ticket {v.capitalize()} {s}",
            f"Richiesta info {s} {d}",
            f"{s} - {d}",
            f"problema urgente {s} non va",
        ]
        title = random.choices(oggetti, weights=pesi_oggetti)[0]

        formati = [
            f"Buongiorno,con la presente si richiede cortesemente di {v} {s} {d}. Restiamo in attesa di un vostro riscontro urgente.Cordiali saluti.",
            f"Ciao, per favore dovremmo {v} {s} {d} il prima possibile.Fammi sapere si ci sono problemi. Grazie!",
            f"Salve, si richiede di {v} {s} {d}. Restiamo in attesa di un riscontro urgente.",
            f"Potete {v} {s} {d}? Grazie, saluti.",
            f"[NOTIFICA AUTOMATICA]: Rilevata necessità di {v} {s} {d}. Si prega di non rispondere a questa email.",
            f"Gentili colleghi,vi ricontatto per sapere se ci sono novità in merito alla richiesta di {v} {s} {d}. Rimango a disposizione. Buon lavoro.",
            f"Purtroppo si è reso necessario {v} {s} {d}.Procedere appena possibile, blocca il lavoro del team.Grazie.",
            f"ciao, bisogna {v} {s} {d} entro oggi se riusciamo... fatemi sapere, grazie",
            f"Per curiosità stavo spulciando {s}, ho scoperto che {v} {s} {d}, e questo è un problema!",
            "ho un problema, quando posso chiamarvi?",
            f"Salve, sono {utente['user_name']}. Ho scoperto che {v} {s}:{d}",
            f"{v} {s} {d}.",
            f"sono {utente['user_name']}, ho un problema critico, ne possiamo parlare il prima possibile?",
            f"scusate ma qui non funziona piu nulla!!! dobbiamo assolutamente {v} {s} o si blocca tutto il reparto!! rispondetemi asap",
            f"Riferimento {s}. Potete verificare? è collegato a {d}. grazie anche da parte del capo.",
        ]
        body = random.choices(formati, weights=pesi_formati)[0]

        # Applichiamo i typo sia al titolo che al corpo
        title = introduci_typo(title, probabilita=0.10)
        body = introduci_typo(body, probabilita=0.20)

        testo_completo = f"{title} {body}".lower()

        conteggio_alta = sum(1 for kw in keywords_alta if kw in testo_completo)
        conteggio_media = sum(1 for kw in keywords_media if kw in testo_completo)

        if conteggio_alta >= 2 and conteggio_media <= 1:
            priority = "Alta"
        elif conteggio_media >= 2 and conteggio_alta <= 1:
            priority = "Media"
        elif conteggio_alta > conteggio_media:
            priority = "Alta"
        elif conteggio_media > 0:
            priority = "Media"
        else:
            priority = "Bassa"

        dataset.append(
            {
                "id": f"TKT-{i+1:04d}",
                "user_name": utente["user_name"],
                "user_email": utente["user_email"],
                "title": title,
                "body": body,
                "category": cat_reale,
                "priority": priority,
            }
        )

    # Scrittura finale ottimizzata
    df = pd.DataFrame(dataset)
    df.to_csv(nome_file_output, index=False, encoding="utf-8-sig")

    print(
        f"Successo! Creato il file {nome_file_output} con {num_ticket} righe totali."
    )

In [21]:
genera_ticket_migliorato("input/lessico.json", pool_utenti, 500,"dataset/ticket_pseudorandom.csv")
df_anteprima = pd.read_csv("dataset/ticket_pseudorandom.csv")
df_anteprima.head(5)["body"]

Successo! Creato il file dataset/ticket_pseudorandom.csv con 500 righe totali.


0    Gentili colleghi,vi ricontatto per sapere se c...
1    ciao, bisogna ripristinare il backup notturno ...
2    ciao, bisogna sollecitare la nota di credito r...
3    Per curiosità stavo spulciando il listino prez...
4    Purtroppo si è reso necessario ripristinare il...
Name: body, dtype: str

# Fase 6: Generazione dataset avanzato tramite LLM (Gemma 4)

Entrambi gli script basati su regole si comportano già abbastanza bene, ma non sono ancora soddisfatto dell'output finale. La pseudorandomizzazione che contradistingue gli script rende i ticket innaturali e, nella maggior parte dei casi, molto ripetitivi. Per superare tale limite, ho deciso dunque di sfruttare la potenza dei modelli generativi.

Invece di optare per un normale e standard prompt all'interno di interfacce web commerciali (come Gemini o soluzioni simili), ho deciso di utilizzare uno strumento molto più versatile e potente, ovvero Gemma4.

Gemma4 è una famiglia di modelli linguistici di grandi dimensione (Large Language Model) sviluppata da Google che ho configurato ed eseguito localmente tramite la piattaforma Ollama. Questa architettura ha permesso di non limitarmi nella scrittura di un prompt per la generazione di n ticket, ma strutturare una funzione che, passati parametri come categoria, priorità, colloquialità, ecc... creasse in **run time** dei prompt casuali per la generazione di **singoli ticket**, per simulare le richieste dei singoli utenti, nella vita reale tutti diversi a modo loro in quanto esseri umani.

Questa soluzione ha inoltre consentito di aggirare i vincoli legati al rate limiting e i costi di consumo tipici delle API basate su cloud (come ad esempio Gemini, anch'essa di Google) e migliorare la resa dell'output.

Dato che considero questa fase un pilastro importante del project work, ho deciso di dedicargli un notebook a parte. Pertanto è stato creato un secondo notebook: **Generazione Dataset Avanzato con Gemma**.

# Fase 7 - Analisi finale dei dataset

Sarà presente la possibilità di eseguire un'EDA più profonda all'interno del terzo file Jupyter Notebook, dove vi sarà la Pipeline di Machine Learning dall'inizio alla fine.

Alla fine di queste celle, nonostante saranno creati due dataframe, verrà conservato all'interno di dataset => ticket_pseudorandom.csv solamente la versione migliorata.

In [22]:
genera_ticket_base("input/lessico.json", pool_utenti, 500,"dataset/ticket_pseudorandom.csv")
df_anteprima = pd.read_csv("dataset/ticket_pseudorandom.csv")
print("*** TICKET BASE ***")
df_anteprima.sample(10)

Successo! Creato il file 'dataset/ticket_pseudorandom.csv' con 500 righe totali.
*** TICKET BASE ***


,id,user_name,user_email,title,body,category,priority
420,TKT-0421,Roberto Matten,r.matten@gmail.com,Estendere la licenza aggiuntiva,Purtroppo si è reso necessario estendere la li...,Commerciale,Bassa
69,TKT-0070,Natale Polignone,natale.polignone@gmail.com,Segnalazione il backup notturno,"Gentili colleghi,vi ricontatto per sapere se c...",Tecnico,Bassa
402,TKT-0403,Pierpaolo Periali,pierpaolo.periali@hotmail.com,Segnalazione il database lento,"Salve, si richiede di ripristinare il database...",Tecnico,Media
460,TKT-0461,Elisa De Dea,elisa_dedea@yahoo.com,URGENTE: Proporre lo sconto commerciale,"Salve, sono Elisa De Dea. Ho scoperto che prop...",Commerciale,Alta
61,TKT-0062,Giulia Doflicher,g.doflicher84@gmail.com,Segnalazione il certificato SSL scaduto,"Salve, sono Giulia Doflicher. Ho scoperto che ...",Tecnico,Alta
5,TKT-0006,Rosa Bellani,rosa.bellani@hotmail.com,Isolare un errore 500,"Buongiorno,con la presente si richiede cortese...",Tecnico,Media
23,TKT-0024,Rocco Colimprain,rocco.colimprain81@outlook.it,Richiesta info il certificato SSL scaduto sull...,"ciao, bisogna ottimizzare il certificato SSL s...",Tecnico,Alta
276,TKT-0277,Gabriele Bignolin,gabriele.bignolin@outlook.it,URGENTE: Ricevere la nota spese,"Buongiorno,con la presente si richiede cortese...",Amministrazione,Alta
347,TKT-0348,Mauro Decilia,mauro_decilia@gmail.com,AIUTO - problema con un bug bloccante,"Gentili colleghi,vi ricontatto per sapere se c...",Tecnico,Media
484,TKT-0485,Nando Grupponio,nando_grupponio@gmail.com,Segnalazione un preventivo,"Salve, si richiede di richiedere un preventivo...",Commerciale,Alta


In [23]:
genera_ticket_migliorato("input/lessico.json", pool_utenti, 500,"dataset/ticket_pseudorandom.csv")
df_anteprima = pd.read_csv("dataset/ticket_pseudorandom.csv")
print("*** TICKET MIGLIORATO ***")
df_anteprima.sample(10)

Successo! Creato il file dataset/ticket_pseudorandom.csv con 500 righe totali.
*** TICKET MIGLIORATO ***


,id,user_name,user_email,title,body,category,priority
399,TKT-0400,Paolo Icardo,paolo_icardo@gmail.com,Segnalare un bug bloccante,Riferimento un bug bloccante. Potete verificar...,Tecnico,Media
186,TKT-0187,Marianna Valpreda,marianna.valpreda82@yahoo.com,URGENTE: Rettificare la fattura,"Salve, si richiede di rettificare la fattura m...",Amministrazione,Media
421,TKT-0422,Santo Stallo,santo.stallo99@gmail.com,URGENTE: Riavviare i permessi di accesso,Purtroppo si è reso necessario riavviare i per...,Tecnico,Alta
390,TKT-0391,Roberto Matten,r.matten@gmail.com,Richiesta supporto - categoria: Amministrazione,"Salve, sono Roberto Matten. Ho scoperto che ar...",Amministrazione,Bassa
359,TKT-0360,Nicola Rinati,nicola.rinati84@gmail.com,URGENTE: Restituire il sollecito di pagamento,Potete restituire il sollecito di pagamento pe...,Amministrazione,Media
314,TKT-0315,Giannino Gandino,gandino.giannino@gmail.com,Ticket Ricevere il rimborso,Riferimento il rimborso. Potete verificare? è ...,Amministrazione,Bassa
276,TKT-0277,Stella Concetti,stella_concetti@gmail.com,problema urgente la licenza aggiuntiva non va,"Salve, si richiede di valutare la licenza aggi...",Commerciale,Media
387,TKT-0388,Mario Dubbini,m.dubbini04@gmail.com,URGENTE: Modificare il listino prezzi aggiornato,"Salve, sono Mario Dubbini. Ho scoperto che mod...",Commerciale,Alta
23,TKT-0024,Nando Grupponio,nando_grupponio@gmail.com,la penale di recesso - per la nuova filiale,"Buongiorno,con la presente si richiede cortese...",Commerciale,Alta
443,TKT-0444,Fulvia Di Ballo,f.diballo72@gmail.com,AIUTO - problema con il pagamento,"Buongiorno,con la presente si richiede cortese...",Amministrazione,Media


# Conclusioni

Avendo consolidato la pipeline di generazione, la base dati elementare è ora pronta per la validazione sul campo. Nel terzo Jupyter Notebook del progetto si procederà all'**addestramento comparativo dei modelli di Machine Learning**. 

L'obiettivo sarà testare l'efficacia del dataset generato per valutare se l'approccio è valido o meno.

Un potenziale miglioramento di questo approccio è sicuramente aggiungere nuove forme lessicali o nuovi formati standard. Sicuramente, nonostante i problemi che riscontrerà nella pipeline, questo approccio risulta scalabile e adattabile a qualsiasi contesto similare. Un qualsiasi esempio potrebbe essere la generazione di dati artificiali riguardo la vendita di prodotti. L'unica accortezza necessaria è inserire un lessico diverso.